# Homework 3 — SQL Exploratory Data Analysis: Aviation (`travel.sqlite`)

**Teodor Simic · Group 11 · Aviation**

**Context.** The client gave us access to its
reservation database. This notebook explores the 8 tables, checks data quality, and
answers business questions with SQLite queries. Pandas is used only to
display the result of each query (`pd.read_sql`); every filter, join, aggregation and
calculation is done inside SQL.

**How to run:** put `travel.sqlite` in the same folder as this notebook, then
`Kernel → Restart & Run All`. The database is opened in **read-only** mode, so the
client's file is never modified. Each query runs in about 2 seconds or less.

## Executive summary
1. **Revenue is concentrated on Moscow and on a few long-haul routes.** Departures from the
   Moscow time zone bring 57% of revenue. The 10 best routes bring about 25% of revenue, led by Moscow ↔ Khabarovsk (≈1.5 bn in both directions).
2. **Prices are fixed, not managed.** On Sochi → Moscow-Sheremetyevo, all 1,090 Business
   tickets were sold at one single price, 3 × Economy. There is room for demand-based
   pricing (yield management).
3. **Small regional aircraft are the least reliable.** The Bombardier CRJ-200 is cancelled
   13 times more often than the Airbus A321, and one aircraft type of the fleet (Airbus
   A320-200) never flies.
4. **Business class is 10% of passengers but 27% of revenue**, and expensive bookings are
   mostly groups (2.0 passengers per booking above 200k vs 1.1 under 20k).
5. **Data limits to clarify with the client:** about 31% of flights have no sale in the
   data, and there is no customer ID (loyalty cannot be measured).

## 1. Setup: connect to the database

In [34]:
import sqlite3
from pathlib import Path
import pandas as pd

db_path = Path("travel.sqlite")
if not db_path.is_file():
    raise FileNotFoundError("Place travel.sqlite in the same folder as this notebook.")

conn = sqlite3.connect("file:travel.sqlite?mode=ro", uri=True)
print("Connected to", db_path.name)

Connected to travel.sqlite


## 2. Exploratory data analysis

### 2.1 Which tables are in the database?

In [35]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

out = pd.read_sql(query, conn)
out

,name
0,aircrafts_data
1,airports_data
2,boarding_passes
3,bookings
4,flights
5,seats
6,ticket_flights
7,tickets


### 2.2 How many rows does each table contain?

In [36]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;", conn)

for table in tables["name"]:
    n_rows = pd.read_sql(f"SELECT COUNT(*) AS n_rows FROM {table};", conn)
    print(table, n_rows["n_rows"][0])

aircrafts_data 9
airports_data 104
boarding_passes 579686
bookings 262788
flights 33121
seats 1339
ticket_flights 1045726
tickets 366733


### 2.3 Columns of the central table, `flights`

In [37]:
schema_info = pd.read_sql("PRAGMA table_info(flights);", conn)
schema_info

,cid,name,type,notnull,dflt_value,pk
0,0,flight_id,INTEGER,1,None,0
1,1,flight_no,character(6),1,None,0
2,2,scheduled_departure,timestamp with time zone,1,None,0
3,3,scheduled_arrival,timestamp with time zone,1,None,0
4,4,departure_airport,character(3),1,None,0
5,5,arrival_airport,character(3),1,None,0
6,6,status,character varying(20),1,None,0
7,7,aircraft_code,character(3),1,None,0
8,8,actual_departure,timestamp with time zone,0,None,0
9,9,actual_arrival,timestamp with time zone,0,None,0


### 2.4 How the tables are linked (keys used in the JOINs)

Each arrow reads: "this column is found in that table".

```text
tickets.book_ref          → bookings.book_ref
ticket_flights.ticket_no  → tickets.ticket_no
ticket_flights.flight_id  → flights.flight_id
boarding_passes.ticket_no → tickets.ticket_no
boarding_passes.flight_id → flights.flight_id
flights.departure_airport → airports_data.airport_code
flights.arrival_airport   → airports_data.airport_code
flights.aircraft_code     → aircrafts_data.aircraft_code
seats.aircraft_code       → aircrafts_data.aircraft_code
```

In words: one booking contains one or several tickets (one per passenger); one
ticket contains one or several flight segments (`ticket_flights`), each with its fare.
Airport and aircraft names are stored in two languages in the same column, so the
notebook identifies airports by their 3-letter code and aircraft by their code.

The aircraft codes used in the rest of the notebook correspond to these models
(9 rows, shown once as a reference):

In [38]:
query = """
SELECT aircraft_code, model, range
FROM aircrafts_data
ORDER BY range DESC;
"""
out = pd.read_sql(query, conn)
out

,aircraft_code,model,range
0,773,"{""en"": ""Boeing 777-300"", ""ru"": ""Боинг 777-300""}",11100
1,763,"{""en"": ""Boeing 767-300"", ""ru"": ""Боинг 767-300""}",7900
2,319,"{""en"": ""Airbus A319-100"", ""ru"": ""Аэробус A319-...",6700
3,320,"{""en"": ""Airbus A320-200"", ""ru"": ""Аэробус A320-...",5700
4,321,"{""en"": ""Airbus A321-200"", ""ru"": ""Аэробус A321-...",5600
5,733,"{""en"": ""Boeing 737-300"", ""ru"": ""Боинг 737-300""}",4200
6,SU9,"{""en"": ""Sukhoi Superjet-100"", ""ru"": ""Сухой Суп...",3000
7,CR2,"{""en"": ""Bombardier CRJ-200"", ""ru"": ""Бомбардье ...",2700
8,CN1,"{""en"": ""Cessna 208 Caravan"", ""ru"": ""Сессна 208...",1200


### 2.5 Period covered by the data

In [39]:
query = """
SELECT MIN(book_date) AS first_booking,
       MAX(book_date) AS last_booking,
       COUNT(*)       AS bookings
FROM bookings;
"""
out = pd.read_sql(query, conn)
out

,first_booking,last_booking,bookings
0,2017-06-21 14:05:00+03,2017-08-15 18:00:00+03,262788


In [40]:
query = """
SELECT MIN(scheduled_departure) AS first_flight,
       MAX(scheduled_departure) AS last_flight,
       COUNT(*)                 AS flights
FROM flights;
"""
out = pd.read_sql(query, conn)
out

,first_flight,last_flight,flights
0,2017-07-16 01:50:00+03,2017-09-14 20:55:00+03,33121


**Finding.** Bookings were made between 21 June and 15 August 2017; flights are
scheduled between 16 July and 14 September 2017. The data is therefore a **snapshot
taken in mid-August 2017**: flights before that date have taken off, flights after it
are still `Scheduled`. Every time-based conclusion below covers about two summer months,
so seasonality cannot be studied.

### 2.6 Join check: does every flight segment match one flight?

Before analysing revenue with JOINs, check that the join neither loses nor duplicates
rows: the joined count must equal the `ticket_flights` count (1,045,726).

In [41]:
query = """
SELECT COUNT(*) AS joined_segments
FROM ticket_flights AS tf
JOIN flights AS f ON tf.flight_id = f.flight_id;
"""
out = pd.read_sql(query, conn)
out

,joined_segments
0,1045726


**Finding.** 1,045,726 rows before and after the join: every segment matches exactly
one flight, so revenue totals computed through this join are reliable.

### 2.7 Can we follow a customer across several trips?

In [42]:
query = """
SELECT COUNT(*)                     AS tickets,
       COUNT(DISTINCT passenger_id) AS distinct_passenger_ids
FROM tickets;
"""
out = pd.read_sql(query, conn)
out

,tickets,distinct_passenger_ids
0,366733,366733


**Finding.** There are as many distinct `passenger_id` values as tickets: no passenger
appears twice. The identifier is a document number attached to one ticket, not a
customer account. **Repeat purchases, loyalty or customer lifetime value cannot be
measured with this database**; the client would need a customer ID to do it.

## 3. Business insights

Amounts are in the currency of the database (roubles). One row of `ticket_flights`
is one **flight segment** sold to one passenger; its `amount` is the segment fare.

### Insight 1 — How much revenue is already earned, and how much is still to be flown?

Revenue of sold segments, split by the status of the flight. Flights that have taken
off (`Arrived`, `Departed`) are earned revenue; the others are revenue already sold
but not yet delivered.

In [43]:
query = """
SELECT f.status,
       COUNT(DISTINCT f.flight_id) AS flights,
       COUNT(*)                    AS segments_sold,
       SUM(tf.amount)              AS revenue,
       ROUND(AVG(tf.amount), 0)    AS avg_fare
FROM ticket_flights AS tf
JOIN flights AS f ON tf.flight_id = f.flight_id
GROUP BY f.status
ORDER BY revenue DESC;
"""
out = pd.read_sql(query, conn)
out

,status,flights,segments_sold,revenue,avg_fare
0,Arrived,11438,574830,11427863000,19880.0
1,Scheduled,10360,446365,8832439600,19787.0
2,On Time,360,20828,406410800,19513.0
3,Departed,40,2275,66583400,29267.0
4,Delayed,28,1428,33684100,23588.0


**Finding.** Flights that have already taken off (`Arrived` + `Departed`) represent
11.49 billion of revenue (11,427,863,000 + 66,583,400). Flights not yet flown (`Scheduled`,
`On Time`, `Delayed`) represent 9.27 bn: **about 45% of the revenue in the database is
future revenue**. No row appears for `Cancelled` flights: they have no sold segment
(see Insight 8).

**Business implication.** Almost half of the revenue is still exposed to operational risk
(delays, cancellations, refunds). Cash planning should treat it as deferred revenue,
not as cash already earned.

### Insight 2 — Which routes generate the most revenue?

A route is a (departure airport, arrival airport) pair. Top 10 by revenue.

In [44]:
query = """
SELECT f.departure_airport, f.arrival_airport,
       COUNT(DISTINCT f.flight_id) AS flights,
       COUNT(*) AS segments_sold,
       SUM(tf.amount) AS revenue,
       ROUND(AVG(tf.amount), 0) AS avg_fare
FROM ticket_flights AS tf
JOIN flights AS f ON tf.flight_id = f.flight_id
GROUP BY f.departure_airport, f.arrival_airport
ORDER BY revenue DESC
LIMIT 10;
"""
out = pd.read_sql(query, conn)
out

,departure_airport,arrival_airport,flights,segments_sold,revenue,avg_fare
0,DME,KHV,61,9647,753478300,78105.0
1,KHV,DME,61,9366,733797800,78347.0
2,DME,OVB,61,15903,548218900,34473.0
3,OVB,DME,61,15402,531503700,34509.0
4,KHV,LED,61,6435,507672400,78892.0
5,LED,KHV,61,6331,498750700,78779.0
6,SVO,OVB,60,13160,458130400,34812.0
7,OVB,SVO,61,12837,446051200,34747.0
8,LED,IKT,60,6958,411603000,59155.0
9,IKT,LED,60,6801,404050300,59410.0


**Finding.** The ranking is made of **5 route pairs flown in both directions** with almost
the same revenue each way (≈61 flights per direction, one per day). Two types of
routes appear:
- **High-fare long-haul routes**: Moscow-Domodedovo (DME) ↔ Khabarovsk (KHV) and
  St Petersburg (LED) ↔ KHV, average fare ≈78,000.
- **High-volume routes**: DME ↔ Novosibirsk (OVB) carries more passengers (15,903 vs 9,647
  from DME to KHV) at less than half the fare (≈34,500).

Together these 10 routes bring about 5.3 billion, **about 25% of total revenue**
(20.77 billion, total of Insight 1).

**Business implication.** A disruption on DME ↔ KHV would hit revenue much more than its
number of flights suggests; these routes deserve priority for aircraft reliability and
crew planning.

### Insight 3 — Revenue by region (time zone of the departure airport)


In [45]:
query = """
SELECT a.timezone AS departure_region,
       COUNT(DISTINCT a.airport_code) AS airports,
       COUNT(DISTINCT f.flight_id) AS flights,
       COUNT(*) AS segments_sold,
       SUM(tf.amount) AS revenue
FROM ticket_flights AS tf
JOIN flights AS f ON tf.flight_id = f.flight_id
JOIN airports_data AS a ON f.departure_airport = a.airport_code
GROUP BY a.timezone
ORDER BY revenue DESC;
"""
out = pd.read_sql(query, conn)
out

,departure_region,airports,flights,segments_sold,revenue
0,Europe/Moscow,39,13320,662365,11880739500
1,Asia/Yekaterinburg,20,4145,170315,2502316900
2,Asia/Vladivostok,3,515,29196,1509955700
3,Asia/Novosibirsk,1,639,46349,1274457100
4,Asia/Irkutsk,4,343,17188,863520800
5,Asia/Krasnoyarsk,7,813,26906,844761700
6,Europe/Samara,5,701,35012,404518700
7,Asia/Sakhalin,1,171,4838,402573800
8,Asia/Yakutsk,4,476,10745,279977300
9,Europe/Volgograd,2,419,17172,194482000


**Finding.** The **Moscow time zone** (39 of the 104 airports) generates 11.88 billion.
**57% of revenue** and 66% of segments sold. The Far East earns much more per flight:
Vladivostok-zone airports bring 1.51 billion with only 515 flights (≈2.9 M per flight) against
≈0.9 M per flight from the Moscow zone, because these flights are long and expensive.

**Business implication.** The network is a **Moscow hub** with long spokes. Dependence on
the Moscow airports is a concentration risk; long Far-East routes are few but very
valuable per flight.

### Insight 4 — Cabin classes: share of passengers vs. share of revenue

In [46]:
query = """
SELECT tf.fare_conditions AS cabin,
       COUNT(*) AS segments_sold,
       SUM(tf.amount) AS revenue,
       ROUND(AVG(tf.amount), 0) AS avg_fare,
       COUNT(DISTINCT f.aircraft_code) AS aircraft_types_offering_it
FROM ticket_flights AS tf
JOIN flights AS f ON tf.flight_id = f.flight_id
GROUP BY tf.fare_conditions
ORDER BY revenue DESC;
"""
out = pd.read_sql(query, conn)
out

,cabin,segments_sold,revenue,avg_fare,aircraft_types_offering_it
0,Economy,920793,14695684400,15960.0,8
1,Business,107642,5505179600,51143.0,6
2,Comfort,17291,566116900,32741.0,1


**Finding.** Economy is 88% of segments sold but only 71% of revenue. **Business is 10% of
segments and 27% of revenue**, with an average fare 3.2 × Economy (51,143 vs 15,960).
Comfort class is marginal (1.7% of segments) and is offered on **one aircraft type
only** (column `aircraft_types_offering_it`).

**Business implication.** Every Business seat sold is worth about three Economy seats.
Adding or protecting Business capacity on the routes where it sells is a stronger revenue
lever than adding Economy seats.

### Insight 5 — Pricing: how many different prices are charged on a busy route?

In [47]:
query = """
SELECT tf.fare_conditions AS cabin,
       COUNT(*) AS segments_sold,
       COUNT(DISTINCT tf.amount) AS distinct_prices,
       MIN(tf.amount) AS min_fare,
       MAX(tf.amount) AS max_fare
FROM ticket_flights AS tf
JOIN flights AS f ON tf.flight_id = f.flight_id
WHERE f.departure_airport = 'AER'
  AND f.arrival_airport = 'SVO'
GROUP BY tf.fare_conditions
ORDER BY max_fare DESC;
"""
out = pd.read_sql(query, conn)
out

,cabin,segments_sold,distinct_prices,min_fare,max_fare
0,Business,1090,1,42100,42100
1,Comfort,1777,1,23900,23900
2,Economy,11794,2,14000,15400


**Finding.** On this route, the 1,090 Business segments were all sold at **one single
price (42,100)**, the 1,777 Comfort segments also at one price (23,900), and Economy at
only two prices (14,000 and 15,400), although bookings were made over almost two months.
Business costs exactly **3 × the lowest Economy fare** (42,100 / 14,000).

**Business implication.** Fares follow a fixed rule, not demand. A revenue-management
tool (prices that rise as the plane fills or as the departure date approaches) is a
clear opportunity for the client.

### Insight 6 — Operations: which aircraft types are cancelled most often?

In [48]:
query = """
SELECT a.aircraft_code,
       a.range AS range_km,
       COUNT(f.flight_id) AS flights,
       SUM(CASE WHEN f.status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled,
       ROUND(100.0 * AVG(CASE WHEN f.status = 'Cancelled' THEN 1 ELSE 0 END), 2) AS cancellation_rate_pct
FROM aircrafts_data AS a
LEFT JOIN flights AS f ON a.aircraft_code = f.aircraft_code
GROUP BY a.aircraft_code, a.range
ORDER BY cancellation_rate_pct DESC;
"""
out = pd.read_sql(query, conn)
out

,aircraft_code,range_km,flights,cancelled,cancellation_rate_pct
0,CR2,2700,9048,181,2.00
1,319,6700,1239,24,1.94
2,CN1,1200,9273,115,1.24
3,733,4200,1274,15,1.18
4,SU9,3000,8504,69,0.81
5,763,7900,1221,6,0.49
6,773,11100,610,1,0.16
7,321,5600,1952,3,0.15
8,320,5700,0,0,0.00


**Finding.** The overall cancellation rate is low (414 of 33,121 flights, 1.25%), but it is
**about 13 times higher on the CRJ-200 (2.00%) than on the Airbus A321 (0.15%)**. Small
regional aircraft (CR2, CN1) and the Airbus A319 (319) are the least reliable. The
`LEFT JOIN` also reveals that the **Airbus A320-200 (code 320) has no flight at all**.

**Business implication.** The CRJ-200 is the first candidate for a fleet review (routes,
maintenance, replacement). An aircraft type that
never flies is an asset costing money (leasing, maintenance): it should be put into
service, leased out or sold.

### Insight 7 — Bookings: do expensive bookings mean expensive tickets, or bigger groups?

In [49]:
query = """
SELECT CASE
           WHEN b.total_amount <  20000 THEN '1. under 20k'
           WHEN b.total_amount <  50000 THEN '2. 20k-50k'
           WHEN b.total_amount < 100000 THEN '3. 50k-100k'
           WHEN b.total_amount < 200000 THEN '4. 100k-200k'
           ELSE '5. 200k and more'
       END AS booking_value_band,
       COUNT(DISTINCT b.book_ref) AS bookings,
       COUNT(*) AS passengers,
       ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT b.book_ref), 2) AS passengers_per_booking
FROM bookings AS b
JOIN tickets AS t ON b.book_ref = t.book_ref
GROUP BY booking_value_band
ORDER BY booking_value_band;
"""
out = pd.read_sql(query, conn)
out

,booking_value_band,bookings,passengers,passengers_per_booking
0,1. under 20k,27740,30930,1.11
1,2. 20k-50k,90566,107072,1.18
2,3. 50k-100k,80008,113996,1.42
3,4. 100k-200k,46943,78938,1.68
4,5. 200k and more,17531,35797,2.04


**Finding.** The number of passengers per booking rises steadily with the booking value:
**1.11 under 20k, 2.04 at 200k and above**. Expensive bookings are therefore mostly
**groups and families**, not only expensive tickets. The 200k+ band contains 17,531
bookings (6.7% of all bookings) but 35,797 passengers.

**Business implication.** The most valuable bookings are made by groups. Group or family
offers (a discount from the 3rd passenger, joint seat selection) target the most
profitable segment.

### Insight 8 — Data coverage: how many flights have no ticket at all in the data?

In [50]:
query = """
SELECT f.status,
       COUNT(DISTINCT f.flight_id)  AS flights,
       COUNT(DISTINCT tf.flight_id) AS flights_with_sales,
       COUNT(DISTINCT f.flight_id) - COUNT(DISTINCT tf.flight_id) AS flights_without_sales,
       ROUND(100.0 * (COUNT(DISTINCT f.flight_id) - COUNT(DISTINCT tf.flight_id))
             / COUNT(DISTINCT f.flight_id), 1) AS pct_without_sales
FROM flights AS f
LEFT JOIN ticket_flights AS tf ON f.flight_id = tf.flight_id
GROUP BY f.status
ORDER BY flights DESC;
"""
out = pd.read_sql(query, conn)
out

,status,flights,flights_with_sales,flights_without_sales,pct_without_sales
0,Arrived,16707,11438,5269,31.5
1,Scheduled,15383,10360,5023,32.7
2,On Time,518,360,158,30.5
3,Cancelled,414,0,414,100.0
4,Departed,58,40,18,31.0
5,Delayed,41,28,13,31.7


**Finding.** About **31–33% of flights have no ticket sold in the database, whatever their
status** (5,269 `Arrived` flights included). All 414 cancelled flights have no sale.
Such a stable share across statuses suggests that the database contains **a sample of
the bookings**, not every passenger; an airline would not fly one flight in three
completely empty.

**Business implication / question for the client.** Before building the final dashboard,
we must ask whether the database is complete. If it is a sample, revenue totals are
**underestimated**, while shares, averages and rankings (Insights 2–7) remain meaningful.

In [26]:
conn.close()
print("Connection closed. The database was opened read-only and was not modified.")

Connection closed. The database was opened read-only and was not modified.
